In [1]:
from dotenv import load_dotenv
from pprint import pprint
import os
from openai import OpenAI
import json
import time

load_dotenv()

# helper function to load json files
def load_json(file_path: str) -> dict:
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)

simple_wiki_data = load_json("simple_wiki_raw_data.json")
normal_wiki_data = load_json("normal_wiki_raw_data.json")


In [ ]:

# ---------- Merging simple and normal page data ----------


# Index both datasets by title (simpler for retrieval)
simple_index = {d['title']: d for d in simple_wiki_data if d.get("title")}
normal_index = {d['title']: d for d in normal_wiki_data if d.get("title")}


# Merge into a list of dicts
wiki_data = [
    {
        "title": title,
        "simple_wiki": {k: v for k, v in simple_index.get(title, {}).items() if k != "title"},
        "normal_wiki": {k: v for k, v in normal_index.get(title, {}).items() if k != "title"},
    }
    for title in set(simple_index) | set(normal_index) # Union of titles in simple and normal wiki data
]

# store in json file
merged_path = 'merged_wiki_data.json'
with open(merged_path, "w", encoding="utf-8") as f:
        json.dump(wiki_data, f, ensure_ascii=False, indent=2)

pprint(wiki_data[:2], width=200, sort_dicts=False)


[{'title': 'Importance sampling',
  'simple_wiki': {'url': 'https://simple.wikipedia.org/wiki/Importance_sampling',
                  'sections': [{'heading': 'Introduction',
                                'paragraphs': ['Importance sampling typically refers to a Monte Carlo method for sampling from a (target) distribution that cannot be sampled from directly. The '
                                               'method works by sampling from a proposal distribution (P(x)), which is ideally similar to the target distribution (Q(x)), and weighting each sample by '
                                               'the ratio of the likelihoods at the sampled point: w(x) = Q(x)/P(x).']}],
                  'categories': ['Randomised_algorithms'],
                  'category_urls': ['https://simple.wikipedia.org/wiki/Category:Randomised_algorithms'],
                  'last_scraped': '2026-02-03T14:22:21.184442+00:00',
                  'first_scraped': '2026-02-03T14:22:21.184442+00:00'},
 

In [ ]:

# ------------ Definition for kids - Generation ------------ 

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.getenv("HF_TOKEN")
)

MAX_INPUT_TOKENS = 16384 # model's (Llama-3.1-8B-Instruct) context length (16384 tokens).

merged_wiki_data = load_json('merged_wiki_data.json')


pages = []

for page in merged_wiki_data:
    # Pick the available source (by default simple_wiki because usually shorter)
    description = (
        page.get("simple_wiki", {}).get("sections") 
        or page.get("normal_wiki", {}).get("sections") 
        or ""
        )
    description = description[:MAX_INPUT_TOKENS]
    
    try:
            completion = client.chat.completions.create(
			model="meta-llama/Llama-3.1-8B-Instruct:novita",
			messages=[
				{"role": "system", "content": "You explain concepts clearly for children around 10 years old. Use simple words, short sentences, and concrete examples. Avoid technical terms unless they are explained. Do not include introductions, titles, or meta commentary. Output only the explanation."},
				{"role": "user", "content": f"Explain this so a 10-year-old can understand it:\n\ntopic: {page['title']}\ndescription: {description}"}
			],
			max_completion_tokens=500,
            )
            simply_explained_10yo = completion.choices[0].message.content
            
    except Exception as e:
        print(f"Skipping {page['title']} due to API error: {e}")
        simply_explained_10yo = None
        
    pages.append({
        "title": page['title'],
        "technical_definition": page.get('normal_wiki', {}).get('sections'), 
        "simple_definition": page.get('simple_wiki', {}).get('sections'),
        "definition4kids": simply_explained_10yo
    })
    
    time.sleep(0.5) # avoid hammering API


# store as json file
full_wiki_data_path = "full_wiki_data.json"
with open(full_wiki_data_path, "w", encoding="utf-8") as f:
    json.dump(pages, f, ensure_ascii=False, indent=2)

# quick check
pprint(pages[:2], width=300, sort_dicts=False)


Skipping Advanced Video Coding due to API error: Error code: 400 - {'code': 400, 'message': "The input (17020 tokens) is longer than the model's context length (16384 tokens).", 'type': 'BadRequestError'}
Skipping Computer security due to API error: Error code: 400 - {'code': 400, 'message': "The input (24252 tokens) is longer than the model's context length (16384 tokens).", 'type': 'BadRequestError'}
Skipping First-order logic due to API error: Error code: 400 - {'code': 400, 'message': "The input (21046 tokens) is longer than the model's context length (16384 tokens).", 'type': 'BadRequestError'}
[{'title': 'Importance sampling',
  'technical_definition': [{'heading': 'Introduction',
                            'paragraphs': ['Importance sampling is a Monte Carlo method for evaluating properties of a particular distribution, while only having samples generated from a different distribution than the distribution of interest. Its introduction in statistics is generally '
             

# Display

In [4]:
from IPython.display import display, Markdown

SPLIT = "\n\n" + "-"*66 + "\n"

def format_sections(title: str, sections: list) -> str:
    """Format a list of sections with a heading and paragraphs"""
    if not sections:
        return ""
    content = [SPLIT + f"## {title}:\n"]  # SPLIT at the start of this section
    for section in sections:
        content.append(f"### {section['heading']}\n")
        content.extend(para + "\n" for para in section.get("paragraphs", []))
    return "\n".join(content)

def pretty_print_page_global(page: dict, definition_type: str | None = None) -> None:
    """
    Print a single page.
    
    definition_type: Optional[str] - one of "technical", "simple", "kids".
                     If None, prints all definitions.
    """
    md = [f"{SPLIT*2}# {page.get('title', 'Untitled')}\n"]

    sections_map = {
        "technical": ("Technical Definition", page.get("technical_definition", [])),
        "simple": ("Simple Definition", page.get("simple_definition", [])),
        "kids": ("Definition Simplified for 10yo", page.get("definition4kids"))
    }

    if definition_type:
        # Only print the selected definition
        if definition_type == "kids" and sections_map["kids"][1]:
            md.append(SPLIT + f"## {sections_map['kids'][0]}:\n{sections_map['kids'][1]}\n")
        elif definition_type in ["technical", "simple"]:
            md.append(format_sections(*sections_map[definition_type]))
        else:
            md.append(f"{SPLIT}**No content found for '{definition_type}'**")
    else:
        # Print all available definitions
        for key, value in sections_map.items():
            if key == "kids" and value[1]:
                md.append(SPLIT + f"## {value[0]}:\n{value[1]}\n")
            elif key in ["technical", "simple"]:
                md.append(format_sections(*value))

    display(Markdown("\n".join(md)))

def pretty_print_all(pages: list, definition_type: str | None = None) -> None:
    """Print every pages"""
    for page in pages:
        pretty_print_page_global(page, definition_type)
    display(Markdown(SPLIT*2))


pretty_print_page_global(pages[4])

pretty_print_all(pages[:2], definition_type='kids')



------------------------------------------------------------------


------------------------------------------------------------------
# Quantum teleportation



------------------------------------------------------------------
## Technical Definition:

### Introduction

Quantum teleportation is a technique for transferring quantum information from a sender at one location to a receiver some distance away. While teleportation is commonly portrayed in science fiction as a means to transfer physical objects from one location to the next, quantum teleportation only transfers quantum information. The sender does not have to know the particular quantum state being transferred. Moreover, the location of the recipient can be unknown, but to complete the quantum teleportation, classical information needs to be sent from sender to receiver. Because classical information needs to be sent, quantum teleportation cannot occur faster than the speed of light.

One of the first scientific articles to investigate quantum teleportation is "Teleporting an Unknown Quantum State via Dual Classical and Einstein–Podolsky–Rosen Channels" published by C. H. Bennett, G. Brassard, C. Crépeau, R. Jozsa, A. Peres, and W. K. Wootters in 1993, in which they proposed using dual communication methods to send/receive quantum information. It was experimentally realized in 1997 by two research groups, led by Sandu Popescu and Anton Zeilinger, respectively.

Experimental determinations of quantum teleportation have been made in information content – including photons, atoms, electrons, and superconducting circuits – as well as distance, with 1,400 km (870 mi) being the longest distance of successful teleportation by Jian-Wei Pan 's team using the Micius satellite for space-based quantum teleportation.

### Non-technical summary

In matters relating to quantum information theory, it is convenient to work with the simplest possible unit of information: the two-state system of the qubit. The qubit functions as the quantum analog of the classic computational part, the bit. Unlike a bit which always acts as either a 0 or a 1, a qubit can behave as a combination of both a 0 and a 1 until the computation is over. The quantum two-state system seeks to transfer quantum information from one location to another location without losing the information and preserving the quality of this information. This process involves moving the information between carriers and not movement of the actual carriers, similar to the traditional process of communications, as two parties remain stationary while the information (digital media, voice, text, etc.) is being transferred, contrary to the implications of the word "teleport". The main components needed for teleportation include a sender, the information (a qubit), a traditional channel, a quantum channel, and a receiver. The sender does not need to know the exact contents of the information being sent. The measurement postulate of quantum mechanics – when a measurement is made upon a quantum state, any subsequent measurements will "collapse" or that the observed state will be lost – creates an imposition within teleportation: if a sender measures their information, the state could collapse when the receiver obtains the data since the state had changed from when the sender made the initial measurement and in so making it different.

For actual teleportation, it is required that an entangled quantum state be created for the qubit to be transferred. Entanglement imposes statistical correlations between otherwise distinct physical systems by creating or placing two or more separate particles into a single, shared quantum state. This intermediate state contains two particles whose quantum states are related to each other: measuring one particle's state provides information about the measurement of the other particle's state. These correlations hold even when measurements are chosen and performed independently, out of causal contact from one another, as verified in Bell test experiments. Thus, an observation resulting from a measurement choice made at one point in spacetime seems to instantaneously affect outcomes in another region, even though light hasn't yet had time to travel the distance, a conclusion seemingly at odds with special relativity. This is known as the EPR paradox. However, such correlations can never be used to transmit any information faster than the speed of light, a statement encapsulated in the no-communication theorem. Thus, teleportation as a whole can never be superluminal, as a qubit cannot be reconstructed until the accompanying classical information arrives.

The sender will combine the particle, whose information is teleported, with one of the entangled particles, causing a change of the overall entangled quantum state. Of this changed state, the particles in the receiver's possession are then sent to an analyzer that will measure the change of the entangled state. The "change" measurement will allow the receiver to recreate the original information that the sender had, resulting in the information being teleported or carried between two people that have different locations. Since the initial quantum information is "destroyed" as it becomes part of the entangled state, the no-cloning theorem is maintained as the information is recreated from the entangled state and not copied during teleportation.

The quantum channel is the communication mechanism that is used for all quantum information transmission and is the channel used for teleportation (relationship of quantum channel to traditional communication channel is akin to the qubit being the quantum analog of the classical bit). However, in addition to the quantum channel, a traditional channel must also be used to accompany a qubit to "preserve" the quantum information. When the change measurement between the original qubit and the entangled particle is made, the measurement result must be carried by a traditional channel so that the quantum information can be reconstructed and the receiver can get the original information. Because of this need for the traditional channel, the speed of teleportation can be no faster than the speed of light (hence the no-communication theorem is not violated). The main advantage with this is that Bell states can be shared using photons from lasers, making teleportation achievable through open space, as there is no need to send information through physical cables or optical fibers.

Quantum states can be encoded in various degrees of freedom of atoms. For example, qubits can be encoded in the degrees of freedom of electrons surrounding the atomic nucleus or in the degrees of freedom of the nucleus itself. Thus, performing this kind of teleportation requires a stock of atoms at the receiving site, available for having qubits imprinted on them.

As of 2015, the quantum states of single photons, photon modes, single atoms, atomic ensembles, defect centers in solids, single electrons, and superconducting circuits have been employed as information bearers.

Understanding quantum teleportation requires a good grounding in finite-dimensional linear algebra, Hilbert spaces and projection matrices. A qubit is described using a two-dimensional complex number -valued vector space (a Hilbert space), which are the primary basis for the formal manipulations given below. A working knowledge of quantum mechanics is not absolutely required to understand the mathematics of quantum teleportation, although without such acquaintance, the deeper meaning of the equations may remain quite mysterious.

### Protocol

The resources required for quantum teleportation are a communication channel capable of transmitting two classical bits, a means of generating an entangled Bell state of qubits and distributing to two different locations, performing a Bell measurement on one of the Bell state qubits, and manipulating the quantum state of the other qubit from the pair. Of course, there must also be some input qubit (in the quantum state ${\displaystyle |\phi \rangle }$ ) to be teleported. The protocol is then as follows:

1) A Bell state is generated with one qubit sent to location A and the other sent to location B. In the picture the alternative name of EPR pair is used for this.
2) A Bell measurement of the Bell state qubit and the qubit to be teleported ( ${\displaystyle |\phi \rangle }$ ) is performed at location A. This yields one of four measurement outcomes which can be encoded in two classical bits of information. Both qubits at location A are then discarded.
3) Using the classical channel, the two bits are sent from A to B. (This is the only potentially time-consuming step after step 1 since information transfer is limited by the speed of light.)
4) As a result of the measurement performed at location A, the Bell state qubit at location B is in one of four possible states. Of these four possible states, one is identical to the original quantum state ${\displaystyle |\phi \rangle }$, and the other three are closely related. The identity of the state actually obtained is encoded in two classical bits and sent to location B. The Bell state qubit at location B is then modified in one of three ways, or not at all, which results in a qubit identical to ${\displaystyle |\phi \rangle }$, the state of the qubit that was chosen for teleportation.

A Bell state is generated with one qubit sent to location A and the other sent to location B. In the picture the alternative name of EPR pair is used for this.

A Bell measurement of the Bell state qubit and the qubit to be teleported ( ${\displaystyle |\phi \rangle }$ ) is performed at location A. This yields one of four measurement outcomes which can be encoded in two classical bits of information. Both qubits at location A are then discarded.

Using the classical channel, the two bits are sent from A to B. (This is the only potentially time-consuming step after step 1 since information transfer is limited by the speed of light.)

As a result of the measurement performed at location A, the Bell state qubit at location B is in one of four possible states. Of these four possible states, one is identical to the original quantum state ${\displaystyle |\phi \rangle }$, and the other three are closely related. The identity of the state actually obtained is encoded in two classical bits and sent to location B. The Bell state qubit at location B is then modified in one of three ways, or not at all, which results in a qubit identical to ${\displaystyle |\phi \rangle }$, the state of the qubit that was chosen for teleportation.

It is worth noticing that the above protocol assumes that the qubits are individually addressable, meaning that the qubits are distinguishable and physically labeled. However, there can be situations where two identical qubits are indistinguishable due to the spatial overlap of their wave functions. Under this condition, the qubits cannot be individually controlled or measured. Nevertheless, a teleportation protocol analogous to that described above can still be (conditionally) implemented by exploiting two independently prepared qubits, with no need of an initial Bell state. This can be made by addressing the internal degrees of freedom of the qubits (e.g., spins or polarisations) by spatially localized measurements performed in separated regions A and B where the two spatially overlapping, indistinguishable qubits can be found. This theoretical prediction has been then verified experimentally via polarized photons in a quantum optical setup.

### Experimental results and records

Work in 1998 verified the initial predictions, and the distance of teleportation was increased in August 2004 to 600 meters, using optical fiber. Subsequently, the record distance for quantum teleportation has been gradually increased to 16 kilometres (9.9 mi), then to 97 km (60 mi), and is now 143 km (89 mi), set in open air experiments in the Canary Islands, done between the two astronomical observatories of the Instituto de Astrofísica de Canarias. There has been a recent record set (as of September 2015 ) using superconducting nanowire detectors that reached the distance of 102 km (63 mi) over optical fiber. For material systems, the record distance is 21 metres (69 ft).

A variant of teleportation called "open-destination" teleportation, with receivers located at multiple locations, was demonstrated in 2004 using five-photon entanglement. Teleportation of a composite state of two single qubits has also been realized. In April 2011, experimenters reported that they had demonstrated teleportation of wave packets of light up to a bandwidth of 10 MHz while preserving strongly nonclassical superposition states. In August 2013, the achievement of "fully deterministic" quantum teleportation, using a hybrid technique, was reported. On 29 May 2014, scientists announced a reliable way of transferring data by quantum teleportation. Quantum teleportation of data had been done before but with highly unreliable methods. On 26 February 2015, scientists at the University of Science and Technology of China in Hefei, led by Chao-yang Lu and Jian-Wei Pan carried out the first experiment teleporting multiple degrees of freedom of a quantum particle. They managed to teleport the quantum information from ensemble of rubidium atoms to another ensemble of rubidium atoms over a distance of 150 metres (490 ft) using entangled photons. In 2016, researchers demonstrated quantum teleportation with two independent sources which are separated by 6.5 km (4.0 mi) in Hefei optical fiber network. In September 2016, researchers at the University of Calgary demonstrated quantum teleportation over the Calgary metropolitan fiber network over a distance of 6.2 km (3.9 mi). In December 2020, as part of the INQNET collaboration, researchers achieved quantum teleportation over a total distance of 44 km (27.3 mi) with fidelities exceeding 90%.

Researchers have also successfully used quantum teleportation to transmit information between clouds of gas atoms, notable because the clouds of gas are macroscopic atomic ensembles.

It is also possible to teleport logical operations, see quantum gate teleportation. In 2018, physicists at Yale demonstrated a deterministic teleported CNOT operation between logically encoded qubits.

First proposed theoretically in 1993, quantum teleportation has since been demonstrated in many different guises. It has been carried out using two-level states of a single photon, a single atom and a trapped ion – among other quantum objects – and also using two photons. In 1997, two groups experimentally achieved quantum teleportation. The first group, led by Sandu Popescu, was based in Italy. An experimental group led by Anton Zeilinger followed a few months later.

The results obtained from experiments done by Popescu's group concluded that classical channels alone could not replicate the teleportation of linearly polarized state and an elliptically polarized state. The Bell state measurement distinguished between the four Bell states, which can allow for a 100% success rate of teleportation, in an ideal representation.

Zeilinger's group produced a pair of entangled photons by implementing the process of parametric down-conversion. In order to ensure that the two photons cannot be distinguished by their arrival times, the photons were generated using a pulsed pump beam. The photons were then sent through narrow-bandwidth filters to produce a coherence time that is much longer than the length of the pump pulse. They then used a two-photon interferometry for analyzing the entanglement so that the quantum property could be recognized when it is transferred from one photon to the other.

Photon 1 was polarized at 45° in the first experiment conducted by Zeilinger's group. Quantum teleportation is verified when both photons are detected in the ${\displaystyle |\Psi ^{-}\rangle _{12}}$ state, which has a probability of 25%. Two detectors, f1 and f2, are placed behind the beam splitter, and recording the coincidence will identify the ${\displaystyle |\Psi ^{-}\rangle _{12}}$ state. If there is a coincidence between detectors f1 and f2, then photon 3 is predicted to be polarized at a 45° angle. Photon 3 is passed through a polarizing beam splitter that selects +45° and −45° polarization. If quantum teleportation has happened, only detector d2, which is at the +45° output, will register a detection. Detector d1, located at the −45° output, will not detect a photon. If there is a coincidence between d2f1f2, with the 45° analysis, and a lack of a d1f1f2 coincidence, with −45° analysis, it is proof that the information from the polarized photon 1 has been teleported to photon 3 using quantum teleportation.

### Quantum teleportation over 143 km

Zeilinger's group developed an experiment using active feed-forward in real time and two free-space optical links, quantum and classical, between the Canary Islands of La Palma and Tenerife, a distance of over 143 kilometers. The results were published in 2012. In order to achieve teleportation, a frequency-uncorrelated polarization-entangled photon pair source, ultra-low-noise single-photon detectors and entanglement assisted clock synchronization were implemented. The two locations were entangled to share the auxiliary state:

$$ {\displaystyle |\Psi ^{-}\rangle _{23}={\frac {1}{\surd 2}}((|H\rangle _{2}|V\rangle _{3})-(|V\rangle _{2}|H\rangle _{3}))} $$

La Palma and Tenerife can be compared to the quantum characters Alice and Bob. Alice and Bob share the entangled state above, with photon 2 being with Alice and photon 3 being with Bob. A third party, Charlie, provides photon 1 (the input photon) which will be teleported to Alice in the generalized polarization state:

$$ {\displaystyle |\phi \rangle _{1}=\alpha |H\rangle _{1}+\beta |V\rangle _{1}} $$

where the complex numbers ${\displaystyle \alpha }$ and ${\displaystyle \beta }$ are unknown to Alice or Bob.

Alice will perform a Bell-state measurement (BSM) that randomly projects the two photons onto one of the four Bell states with each one having a probability of 25%. Photon 3 will be projected onto ${\displaystyle |\phi \rangle }$, the input state. Alice transmits the outcome of the BSM to Bob, via the classical channel, where Bob is able to apply the corresponding unitary operation to obtain photon 3 in the initial state of photon 1. Bob will not have to do anything if he detects the ${\displaystyle |\psi ^{-}\rangle _{12}}$ state. Bob will need to apply a ${\displaystyle \pi }$ phase shift to photon 3 between the horizontal and vertical component if the ${\displaystyle |\psi ^{+}\rangle _{12}}$ state is detected.

The results of Zeilinger's group concluded that the average fidelity (overlap of the ideal teleported state with the measured density matrix) was 0.863 with a standard deviation of 0.038. The link attenuation during their experiments varied between 28.1 dB and 39.0 dB, which was a result of strong winds and rapid temperature changes. Despite the high loss in the quantum free-space channel, the average fidelity surpassed the classical limit of 2/3. Therefore, Zeilinger's group successfully demonstrated quantum teleportation over a distance of 143 km.

### Quantum teleportation across the Danube River

In 2004, a quantum teleportation experiment was conducted across the Danube River in Vienna, a total of 600 meters. An 800-meter-long optical fiber wire was installed in a public sewer system underneath the Danube River, and it was exposed to temperature changes and other environmental influences. Alice must perform a joint Bell state measurement (BSM) on photon b, the input photon, and photon c, her part of the entangled photon pair (photons c and d). Photon d, Bob's receiver photon, will contain all of the information on the input photon b, except for a phase rotation that depends on the state that Alice observed. This experiment implemented an active feed-forward system that sends Alice's measurement results via a classical microwave channel with a fast electro-optical modulator in order to exactly replicate Alice's input photon. The teleportation fidelity obtained from the linear polarization state at 45° varied between 0.84 and 0.90, which is well above the classical fidelity limit of 0.66.

### Deterministic quantum teleportation with atoms

Three qubits are required for this process: the source qubit from the sender, the ancillary qubit, and the receiver's target qubit, which is maximally entangled with the ancillary qubit. For this experiment, ${\displaystyle {\ce {^{40}Ca+}}}$ ions were used as the qubits. Ions 2 and 3 are prepared in the Bell state ${\displaystyle |\psi ^{+}\rangle _{23}={\frac {1}{\sqrt {2}}}(|0\rangle _{2}|1\rangle _{3}+|1\rangle _{2}|0\rangle _{3})}$. The state of ion 1 is prepared arbitrarily. The quantum states of ions 1 and 2 are measured by illuminating them with light at a specific wavelength. The obtained fidelities for this experiment ranged between 73% and 76%. This is larger than the maximum possible average fidelity of 66.7% that can be obtained using completely classical resources.

### Ground-to-satellite quantum teleportation

The quantum state being teleported in this experiment is ${\displaystyle |\chi \rangle _{1}=\alpha |H\rangle _{1}+\beta |V\rangle _{1}}$, where ${\displaystyle \alpha }$ and ${\displaystyle \beta }$ are unknown complex numbers, ${\displaystyle |H\rangle }$ represents the horizontal polarization state, and ${\displaystyle |V\rangle }$ represents the vertical polarization state. The qubit prepared in this state is generated in a laboratory in Ngari, Tibet. The goal was to teleport the quantum information of the qubit to the Micius satellite that was launched on August 16, 2016, at an altitude of around 500 km. When a Bell state measurement is conducted on photons 1 and 2 and the resulting state is ${\displaystyle |\phi ^{+}\rangle _{12}={\frac {1}{\sqrt {2}}}(|H\rangle _{1}|H\rangle _{2}+|V\rangle _{1}|V\rangle _{2}))}$, photon 3 carries this desired state. If the Bell state detected is ${\displaystyle |\phi ^{-}\rangle _{12}={\frac {1}{\sqrt {2}}}(|H\rangle _{1}|H\rangle _{2}-|V\rangle _{1}|V\rangle _{2})}$, then a phase shift of ${\displaystyle \pi }$ is applied to the state to get the desired quantum state. The distance between the ground station and the satellite changes from as little as 500 km to as large as 1,400 km. Because of the changing distance, the channel loss of the uplink varies between 41 dB and 52 dB. The average fidelity obtained from this experiment was 0.80 with a standard deviation of 0.01. Therefore, this experiment successfully established a ground-to-satellite uplink over a distance of 500–1,400 km using quantum teleportation. This is an essential step towards creating a global-scale quantum internet.

### Quantum teleportation over internet cables

Quantum teleportation has been demonstrated over fiber optic cables simultaneously carrying regular telecommunications traffic. This eliminates the need for separate, dedicated infrastructure for quantum networking and shows that quantum teleportation and classical communications can coexist on the same fiber optic cables. A less crowded wavelength of light was used for the quantum signal and special filters were required to reduce noise from other traffic.

### Quantum teleportation with nonlinear sum frequency generation

In April 2025, researchers at the University of Illinois Urbana-Champaign achieved quantum teleportation with 94% fidelity using a nanophotonic indium - gallium - phosphide platform to perform nonlinear sum frequency generation (SFG). This method mitigated multiphoton noise and boosted teleportation efficiency by a factor of 10,000 compared to prior SFG-based systems.

### Formal presentation

There are a variety of ways in which the teleportation protocol can be written mathematically. Some are very compact but abstract, and some are verbose but straightforward and concrete. The presentation below is of the latter form: verbose, but has the benefit of showing each quantum state simply and directly. Later sections review more compact notations.

The teleportation protocol begins with a quantum state or qubit ${\displaystyle |\psi \rangle }$, in Alice's possession, that she wants to convey to Bob. This qubit can be written generally, in bra–ket notation, as:

$$ {\displaystyle |\psi \rangle _{C}=\alpha |0\rangle _{C}+\beta |1\rangle _{C}.} $$

The subscript C above is used only to distinguish this state from A and B, below.

Next, the protocol requires that Alice and Bob share a maximally entangled state. This state is fixed in advance, by mutual agreement between Alice and Bob, and can be any one of the four Bell states shown. It does not matter which one.

${\displaystyle |\Phi ^{+}\rangle _{AB}={\frac {1}{\sqrt {2}}}(|0\rangle _{A}\otimes |0\rangle _{B}+|1\rangle _{A}\otimes |1\rangle _{B})}$,

${\displaystyle |\Psi ^{+}\rangle _{AB}={\frac {1}{\sqrt {2}}}(|0\rangle _{A}\otimes |1\rangle _{B}+|1\rangle _{A}\otimes |0\rangle _{B})}$,

${\displaystyle |\Psi ^{-}\rangle _{AB}={\frac {1}{\sqrt {2}}}(|0\rangle _{A}\otimes |1\rangle _{B}-|1\rangle _{A}\otimes |0\rangle _{B})}$.

${\displaystyle |\Phi ^{-}\rangle _{AB}={\frac {1}{\sqrt {2}}}(|0\rangle _{A}\otimes |0\rangle _{B}-|1\rangle _{A}\otimes |1\rangle _{B})}$,

In the following, assume that Alice and Bob share the state ${\displaystyle |\Phi ^{+}\rangle _{AB}.}$ Alice obtains one of the particles in the pair, with the other going to Bob. (This is implemented by preparing the particles together and shooting them to Alice and Bob from a common source.) The subscripts A and B in the entangled state refer to Alice's or Bob's particle.

At this point, Alice has two particles ( C, the one she wants to teleport, and A, one of the entangled pair), and Bob has one particle, B. In the total system, the state of these three particles is given by

$$ {\displaystyle |\psi \rangle _{C}\otimes |\Phi ^{+}\rangle _{AB}=(\alpha |0\rangle _{C}+\beta |1\rangle _{C})\otimes {\frac {1}{\sqrt {2}}}(|0\rangle _{A}\otimes |0\rangle _{B}+|1\rangle _{A}\otimes |1\rangle _{B}).} $$

Alice will then make a local measurement in the Bell basis (i.e. the four Bell states) on the two particles in her possession. To make the result of her measurement clear, it is best to write the state of Alice's two qubits as superpositions of the Bell basis. This is done by using the following general identities, which are easily verified:

$$ {\displaystyle |0\rangle \otimes |0\rangle ={\frac {1}{\sqrt {2}}}(|\Phi ^{+}\rangle +|\Phi ^{-}\rangle ),} $$

$$ {\displaystyle |0\rangle \otimes |1\rangle ={\frac {1}{\sqrt {2}}}(|\Psi ^{+}\rangle +|\Psi ^{-}\rangle ),} $$

$$ {\displaystyle |1\rangle \otimes |0\rangle ={\frac {1}{\sqrt {2}}}(|\Psi ^{+}\rangle -|\Psi ^{-}\rangle ),} $$

and

$$ {\displaystyle |1\rangle \otimes |1\rangle ={\frac {1}{\sqrt {2}}}(|\Phi ^{+}\rangle -|\Phi ^{-}\rangle ).} $$

After expanding the expression for ${\textstyle {\begin{aligned}|&\psi \rangle _{C}\otimes \ |\Phi ^{+}\rangle _{AB}\end{aligned}}}$, one applies these identities to the qubits with A and C subscripts. In particular, ${\displaystyle \alpha {\frac {1}{\sqrt {2}}}|0\rangle _{C}\otimes |0\rangle _{A}\otimes |0\rangle _{B}=\alpha {\frac {1}{2}}(|\Phi ^{+}\rangle _{CA}+|\Phi ^{-}\rangle _{CA})\otimes |0\rangle _{B},}$ and the other terms follow similarly. Combining similar terms, the total three particle state of A, B and C together becomes the following four-term superposition:

$$ {\displaystyle {\begin{aligned}|&\psi \rangle _{C}\otimes \ |\Phi ^{+}\rangle _{AB}\ =\\{\frac {1}{2}}{\Big \lbrack }\ &|\Phi ^{+}\rangle _{CA}\otimes (\alpha |0\rangle _{B}+\beta |1\rangle _{B})\ +\ |\Phi ^{-}\rangle _{CA}\otimes (\alpha |0\rangle _{B}-\beta |1\rangle _{B})\\\ +\ &|\Psi ^{+}\rangle _{CA}\otimes (\alpha |1\rangle _{B}+\beta |0\rangle _{B})\ +\ |\Psi ^{-}\rangle _{CA}\otimes (\alpha |1\rangle _{B}-\beta |0\rangle _{B}){\Big \rbrack }.\\\end{aligned}}} $$

Note that all three particles are still in the same total state since no operations have been performed. Rather, the above is just a change of basis on Alice's part of the system. This change has moved the entanglement from particles A and B to particles C and A. The actual teleportation occurs when Alice measures her two qubits (C and A) in the Bell basis

$$ {\displaystyle |\Phi ^{+}\rangle _{CA},|\Phi ^{-}\rangle _{CA},|\Psi ^{+}\rangle _{CA},|\Psi ^{-}\rangle _{CA}.} $$

Equivalently, the measurement may be done in the computational basis, ${\displaystyle \{|0\rangle,|1\rangle \}}$, by mapping each Bell state uniquely to one of ${\displaystyle \{|0\rangle \otimes |0\rangle,|0\rangle \otimes |1\rangle,|1\rangle \otimes |0\rangle,|1\rangle \otimes |1\rangle \}}$ with the quantum circuit in the figure to the right.

The result of Alice's (local) measurement is a collection of two classical bits (00, 01, 10 or 11) related to one of the following four states (with equal probability of 1/4), after the three-particle state has collapsed into one of the states:

- ${\displaystyle |\Phi ^{+}\rangle _{CA}\otimes (\alpha |0\rangle _{B}+\beta |1\rangle _{B})}$
- ${\displaystyle |\Phi ^{-}\rangle _{CA}\otimes (\alpha |0\rangle _{B}-\beta |1\rangle _{B})}$
- ${\displaystyle |\Psi ^{+}\rangle _{CA}\otimes (\alpha |1\rangle _{B}+\beta |0\rangle _{B})}$
- ${\displaystyle |\Psi ^{-}\rangle _{CA}\otimes (\alpha |1\rangle _{B}-\beta |0\rangle _{B})}$

${\displaystyle |\Phi ^{+}\rangle _{CA}\otimes (\alpha |0\rangle _{B}+\beta |1\rangle _{B})}$

${\displaystyle |\Phi ^{-}\rangle _{CA}\otimes (\alpha |0\rangle _{B}-\beta |1\rangle _{B})}$

${\displaystyle |\Psi ^{+}\rangle _{CA}\otimes (\alpha |1\rangle _{B}+\beta |0\rangle _{B})}$

${\displaystyle |\Psi ^{-}\rangle _{CA}\otimes (\alpha |1\rangle _{B}-\beta |0\rangle _{B})}$

Alice's two particles are now entangled to each other, in one of the four Bell states, and the entanglement originally shared between Alice's and Bob's particles is now broken. Bob's particle takes on one of the four superposition states shown above. Note how Bob's qubit is now in a state that resembles the state to be teleported. The four possible states for Bob's qubit are unitary images of the state to be teleported.

The result of Alice's Bell measurement tells her which of the above four states the system is in. She can now send her result to Bob through a classical channel. Two classical bits can communicate which of the four results she obtained. After Bob receives the message from Alice, he will know which of the four states his particle is in. Using this information, he performs a unitary operation on his particle to transform it to the desired state ${\displaystyle \alpha |0\rangle _{B}+\beta |1\rangle _{B}}$:

- If Alice indicates her result is ${\displaystyle |\Phi ^{+}\rangle _{CA}}$, Bob knows his qubit is already in the desired state and does nothing. This amounts to the trivial unitary operation, the identity operator.
- If the message indicates ${\displaystyle |\Phi ^{-}\rangle _{CA}}$, Bob would send his qubit through the unitary quantum gate given by the Pauli matrix

If Alice indicates her result is ${\displaystyle |\Phi ^{+}\rangle _{CA}}$, Bob knows his qubit is already in the desired state and does nothing. This amounts to the trivial unitary operation, the identity operator.

If the message indicates ${\displaystyle |\Phi ^{-}\rangle _{CA}}$, Bob would send his qubit through the unitary quantum gate given by the Pauli matrix

$$ {\displaystyle \sigma _{3}={\begin{bmatrix}1&0\\0&-1\end{bmatrix}}} $$

to recover the state.

- If Alice's message corresponds to ${\displaystyle |\Psi ^{+}\rangle _{CA}}$, Bob applies the gate

If Alice's message corresponds to ${\displaystyle |\Psi ^{+}\rangle _{CA}}$, Bob applies the gate

$$ {\displaystyle \sigma _{1}={\begin{bmatrix}0&1\\1&0\end{bmatrix}}} $$

to his qubit.

- Finally, for the remaining case, the appropriate gate is given by

Finally, for the remaining case, the appropriate gate is given by

$$ {\displaystyle \sigma _{3}\sigma _{1}=-\sigma _{1}\sigma _{3}=i\sigma _{2}={\begin{bmatrix}0&1\\-1&0\end{bmatrix}}.} $$

Teleportation is thus achieved. The above-mentioned three gates correspond to rotations of π radians (180°) about appropriate axes (X, Y and Z) in the Bloch sphere picture of a qubit.

Some remarks:

- After this operation, Bob's qubit will take on the state ${\displaystyle |\psi \rangle _{B}=\alpha |0\rangle _{B}+\beta |1\rangle _{B}}$, and Alice's qubit becomes an (undefined) part of an entangled state. Teleportation does not result in the copying of qubits, and hence is consistent with the no-cloning theorem.
- There is no transfer of matter or energy involved. Alice's particle has not been physically moved to Bob; only its state has been transferred. The term "teleportation", coined by Bennett, Brassard, Crépeau, Jozsa, Peres and Wootters, reflects the indistinguishability of quantum mechanical particles.
- For every qubit teleported, Alice needs to send Bob two classical bits of information. These two classical bits do not carry complete information about the qubit being teleported. If an eavesdropper intercepts the two bits, she may know exactly what Bob needs to do in order to recover the desired state. However, this information is useless if she cannot interact with the entangled particle in Bob's possession.

After this operation, Bob's qubit will take on the state ${\displaystyle |\psi \rangle _{B}=\alpha |0\rangle _{B}+\beta |1\rangle _{B}}$, and Alice's qubit becomes an (undefined) part of an entangled state. Teleportation does not result in the copying of qubits, and hence is consistent with the no-cloning theorem.

There is no transfer of matter or energy involved. Alice's particle has not been physically moved to Bob; only its state has been transferred. The term "teleportation", coined by Bennett, Brassard, Crépeau, Jozsa, Peres and Wootters, reflects the indistinguishability of quantum mechanical particles.

For every qubit teleported, Alice needs to send Bob two classical bits of information. These two classical bits do not carry complete information about the qubit being teleported. If an eavesdropper intercepts the two bits, she may know exactly what Bob needs to do in order to recover the desired state. However, this information is useless if she cannot interact with the entangled particle in Bob's possession.

### Certifying quantum teleportation

When implementing the quantum teleportation protocol, different experimental noises may arise affecting the state transference. The usual way to benchmark a particular teleportation procedure is by using the average fidelity: Given an arbitrary teleportation protocol producing output states ${\displaystyle \rho _{i}}$ with probability ${\displaystyle p_{i}}$ for an initial state ${\displaystyle \rho =|\psi \rangle \langle \psi |}$, the average fidelity is defined as:

$$ {\displaystyle \langle {\overline {F}}\rangle =\int \sum _{i}p_{i}F(\rho,\rho _{i})d\psi } $$

where the integration is performed over the Haar measure defined by assuming maximal uncertainty over the initial quantum states ${\displaystyle |\psi \rangle }$, and ${\displaystyle F(\rho,\rho _{i})=\left({\text{Tr}}{\sqrt {{\sqrt {\rho }}\rho _{i}{\sqrt {\rho }}}}\right)^{2}}$ is the Uhlmann-Jozsa fidelity.

The widely known classical threshold is obtained by optimizing the average fidelity over all classical protocols (i.e. when the sender Alice and the receiver Bob can use just a classical channel to communicate with each other). When teleportation involves qubit states, the maximal classical average fidelity is ${\displaystyle 2/3}$. In this way, a particular protocol with average fidelity ${\displaystyle \langle {\overline {F}}\rangle }$ is certified as useful if ${\displaystyle \langle {\overline {F}}\rangle \geq 2/3}$.

However, using the Uhlmann-Jozsa fidelity as the unique distance measure for benchmarking teleportation is not justified, and one may choose different distinguishability measures. For example, there may exist reasons depending on the context in which other measures might be more suitable than fidelity. In this way, the average distance of teleportation is defined as:

$$ {\displaystyle \langle {\overline {D}}\rangle =\int \sum _{i}p_{i}D(\rho,\rho _{i})d\psi } $$

being ${\displaystyle D(\rho,\sigma )}$ a well-behaved (i.e. satisfying identity of indiscernibles and unitary invariance) distinguishability measure between quantum states. Consequently, different classical thresholds exist, depending on the considered distance measure (classical thresholds for Trace distance, quantum Jensen–Shannon divergence, transmission distance, Bures distance, wootters distance, and quantum Hellinger distance, among others, were obtained in Ref. ). This points out a particular issue when certifying quantum teleportation: Given a teleportation protocol, its certification is not a universal fact in the sense that depends on the distance used. Then, a particular protocol might be certified as useful for a set of distance quantifiers, and non-useful for other distinguishability measures.

### Alternative notations

There are a variety of different notations in use that describe the teleportation protocol. One common one is by using the notation of quantum gates.

In the above derivation, the unitary transformation that is the change of basis (from the standard product basis into the Bell basis) can be written using quantum gates. Direct calculation shows that this gate is given by

$$ {\displaystyle G=(H\otimes I)\operatorname {CNOT} } $$

where H is the one qubit Walsh-Hadamard gate and ${\displaystyle \operatorname {CNOT} }$ is the Controlled NOT gate.

### Entanglement swapping

Teleportation can be applied not just to pure states, but also mixed states, that can be regarded as the state of a single subsystem of an entangled pair. The so-called entanglement swapping is a simple and illustrative example.

If Alice and Bob share an entangled pair, and Bob teleports his particle to Carol, then Alice's particle is now entangled with Carol's particle. This situation can also be viewed symmetrically as follows:

Alice and Bob share an entangled pair, and Bob and Carol share a different entangled pair. Now let Bob perform a projective measurement on his two particles in the Bell basis and communicate the result to Carol. These actions are precisely the teleportation protocol described above with Bob's first particle, the one entangled with Alice's particle, as the state to be teleported. When Carol finishes the protocol she now has a particle with the teleported state, that is an entangled state with Alice's particle. Thus, although Alice and Carol never interacted with each other, their particles are now entangled.

A detailed diagrammatic derivation of entanglement swapping has been given by Bob Coecke, presented in terms of categorical quantum mechanics.

### Algorithm for swapping Bell pairs

An important application of entanglement swapping is distributing Bell states for use in entanglement distributed quantum networks. A technical description of the entanglement swapping protocol is given here for pure Bell states.

1) Alice and Bob locally prepare known Bell pairs resulting in the initial state: ${\displaystyle |\psi \rangle _{\rm {in}}=|\Phi ^{+}\rangle _{A_{1},A_{2}}|\Phi ^{+}\rangle _{B_{1},B_{2}}}$
2) Alice sends qubit ${\displaystyle A_{1}}$ to a third party Carol
3) Bob sends qubit ${\displaystyle B_{1}}$ to Carol
4) Carol performs a Bell projection between ${\displaystyle A_{1}}$ and ${\displaystyle B_{1}}$ that by chance (all four Bell states are possible and recognizable) results in the measurement outcome: ${\displaystyle \langle \Phi ^{+}|_{A_{1},B_{1}}|\psi \rangle _{\rm {in}}=|\Phi ^{+}\rangle _{A_{2},B_{2}}}$
5) In the case of the other three Bell projection outcomes, local corrections given by Pauli operators are made by Alice and or Bob after Carol has communicated the results of the measurement. ${\displaystyle \langle \Phi ^{-}|_{A_{1},B_{1}}|\psi \rangle _{\rm {in}}={\hat {Z}}_{B_{2}}|\Phi ^{+}\rangle _{A_{2},B_{2}}}$ ${\displaystyle \langle \Psi ^{+}|_{A_{1},B_{1}}|\psi \rangle _{\rm {in}}={\hat {X}}_{B_{2}}|\Phi ^{+}\rangle _{A_{2},B_{2}}}$ ${\displaystyle \langle \Psi ^{-}|_{A_{1},B_{1}}|\psi \rangle _{\rm {in}}={\hat {X}}_{B_{2}}{\hat {Z}}_{B_{2}}|\Phi ^{+}\rangle _{A_{2},B_{2}}}$
6) Alice and Bob now have a Bell pair between qubits ${\displaystyle A_{2}}$ and ${\displaystyle B_{2}}$ ${\displaystyle |\psi \rangle _{\rm {out}}=|\Phi ^{+}\rangle _{A_{2},B_{2}}}$

Alice and Bob locally prepare known Bell pairs resulting in the initial state: ${\displaystyle |\psi \rangle _{\rm {in}}=|\Phi ^{+}\rangle _{A_{1},A_{2}}|\Phi ^{+}\rangle _{B_{1},B_{2}}}$

Alice sends qubit ${\displaystyle A_{1}}$ to a third party Carol

Bob sends qubit ${\displaystyle B_{1}}$ to Carol

Carol performs a Bell projection between ${\displaystyle A_{1}}$ and ${\displaystyle B_{1}}$ that by chance (all four Bell states are possible and recognizable) results in the measurement outcome: ${\displaystyle \langle \Phi ^{+}|_{A_{1},B_{1}}|\psi \rangle _{\rm {in}}=|\Phi ^{+}\rangle _{A_{2},B_{2}}}$

In the case of the other three Bell projection outcomes, local corrections given by Pauli operators are made by Alice and or Bob after Carol has communicated the results of the measurement. ${\displaystyle \langle \Phi ^{-}|_{A_{1},B_{1}}|\psi \rangle _{\rm {in}}={\hat {Z}}_{B_{2}}|\Phi ^{+}\rangle _{A_{2},B_{2}}}$ ${\displaystyle \langle \Psi ^{+}|_{A_{1},B_{1}}|\psi \rangle _{\rm {in}}={\hat {X}}_{B_{2}}|\Phi ^{+}\rangle _{A_{2},B_{2}}}$ ${\displaystyle \langle \Psi ^{-}|_{A_{1},B_{1}}|\psi \rangle _{\rm {in}}={\hat {X}}_{B_{2}}{\hat {Z}}_{B_{2}}|\Phi ^{+}\rangle _{A_{2},B_{2}}}$

Alice and Bob now have a Bell pair between qubits ${\displaystyle A_{2}}$ and ${\displaystyle B_{2}}$ ${\displaystyle |\psi \rangle _{\rm {out}}=|\Phi ^{+}\rangle _{A_{2},B_{2}}}$

### Generalizations of the teleportation protocol

The basic teleportation protocol for a qubit described above has been generalized in several directions, in particular regarding the dimension of the system teleported and the number of parties involved (either as sender, controller, or receiver).

### d -dimensional systems

A generalization to ${\displaystyle d}$ -level systems (so-called qudits ) is straight forward and was already discussed in the original paper by Bennett et al.: the maximally entangled state of two qubits has to be replaced by a maximally entangled state of two qudits and the Bell measurement by a measurement defined by a maximally entangled orthonormal basis. All possible such generalizations were discussed by Werner in 2001.

The generalization to infinite-dimensional so-called continuous-variable systems was proposed by Braunstein and Kimble and led to the first teleportation experiment that worked unconditionally.

### Multipartite versions

The use of multipartite entangled states instead of a bipartite maximally entangled state allows for several new features: either the sender can teleport information to several receivers either sending the same state to all of them (which allows to reduce the amount of entanglement needed for the process) or teleporting multipartite states or sending a single state in such a way that the receiving parties need to cooperate to extract the information. A different way of viewing the latter setting is that some of the parties can control whether the others can teleport.

### Logic gate teleportation

In general, mixed states ρ may be transported, and a linear transformation ω applied during teleportation, thus allowing data processing of quantum information. This is one of the foundational building blocks of quantum information processing. This is demonstrated below.

### General description

A general teleportation scheme can be described as follows. Three quantum systems are involved. System 1 is the (unknown) state ρ to be teleported by Alice. Systems 2 and 3 are in a maximally entangled state ω that are distributed to Alice and Bob, respectively. The total system is then in the state

$$ {\displaystyle \rho \otimes \omega.} $$

A successful teleportation process is a LOCC quantum channel Φ that satisfies

$$ {\displaystyle (\operatorname {Tr} _{12}\circ \Phi )(\rho \otimes \omega )=\rho \,,} $$

where Tr 12 is the partial trace operation with respect systems 1 and 2, and ${\displaystyle \circ }$ denotes the composition of maps. This describes the channel in the Schrödinger picture.

Taking adjoint maps in the Heisenberg picture, the success condition becomes

$$ {\displaystyle \langle \Phi (\rho \otimes \omega )|I\otimes O\rangle =\langle \rho |O\rangle } $$

for all observable O on Bob's system. The tensor factor in ${\displaystyle I\otimes O}$ is ${\displaystyle 12\otimes 3}$ while that of ${\displaystyle \rho \otimes \omega }$ is ${\displaystyle 1\otimes 23}$.

### Further details

The proposed channel Φ can be described more explicitly. To begin teleportation, Alice performs a local measurement on the two subsystems (1 and 2) in her possession. Assume the local measurement have effects

$$ {\displaystyle {F_{i}}={M_{i}^{2}}.} $$

If the measurement registers the i -th outcome, the overall state collapses to

$$ {\displaystyle (M_{i}\otimes I)(\rho \otimes \omega )(M_{i}\otimes I).} $$

The tensor factor in ${\displaystyle (M_{i}\otimes I)}$ is ${\displaystyle 12\otimes 3}$ while that of ${\displaystyle \rho \otimes \omega }$ is ${\displaystyle 1\otimes 23}$. Bob then applies a corresponding local operation Ψ i on system 3. On the combined system, this is described by

$$ {\displaystyle (Id\otimes \Psi _{i})(M_{i}\otimes I)(\rho \otimes \omega )(M_{i}\otimes I).} $$

where Id is the identity map on the composite system ${\displaystyle 1\otimes 2}$.

Therefore, the channel Φ is defined by

$$ {\displaystyle \Phi (\rho \otimes \omega )=\sum _{i}(Id\otimes \Psi _{i})(M_{i}\otimes I)(\rho \otimes \omega )(M_{i}\otimes I)} $$

Notice Φ satisfies the definition of LOCC. As stated above, the teleportation is said to be successful if, for all observable O on Bob's system, the equality

$$ {\displaystyle \langle \Phi (\rho \otimes \omega ),I\otimes O\rangle =\langle \rho,O\rangle } $$

holds. The left hand side of the equation is:

$$ {\displaystyle \sum _{i}\langle (Id\otimes \Psi _{i})(M_{i}\otimes I)(\rho \otimes \omega )(M_{i}\otimes I),\;I\otimes O\rangle } $$

$$ {\displaystyle =\sum _{i}\langle (M_{i}\otimes I)(\rho \otimes \omega )(M_{i}\otimes I),\;I\otimes \Psi _{i}^{*}(O)\rangle } $$

where Ψ i * is the adjoint of Ψ i in the Heisenberg picture. Assuming all objects are finite dimensional, this becomes

$$ {\displaystyle \sum _{i}\operatorname {Tr} \;(\rho \otimes \omega )(F_{i}\otimes \Psi _{i}^{*}(O)).} $$

The success criterion for teleportation has the expression

$$ {\displaystyle \sum _{i}\operatorname {Tr} \;(\rho \otimes \omega )(F_{i}\otimes \Psi _{i}^{*}(O))=\operatorname {Tr} \;\rho \cdot O.} $$

### Local explanation of the phenomenon

A local explanation of quantum teleportation is put forward by David Deutsch and Patrick Hayden, with respect to the many-worlds interpretation of quantum mechanics. Their paper asserts that the two bits that Alice sends Bob contain "locally inaccessible information" resulting in the teleportation of the quantum state. "The ability of quantum information to flow through a classical channel [...], surviving decoherence, is [...] the basis of quantum teleportation."

### Recent developments

While quantum teleportation is in an infancy stage, there are many aspects pertaining to teleportation that scientists are working to better understand or improve the process that include:

### Higher dimensions

Quantum teleportation can improve the errors associated with fault tolerant quantum computation via an arrangement of logic gates. Experiments by D. Gottesman and I. L. Chuang have determined that a "Clifford hierarchy" gate arrangement which acts to enhance protection against environmental errors. Overall, a higher threshold of error is allowed with the Clifford hierarchy as the sequence of gates requires less resources that are needed for computation. While the more gates that are used in a quantum computer create more noise, the gates arrangement and use of teleportation in logic transfer can reduce this noise as it calls for less "traffic" that is compiled in these quantum networks. The more qubits used for a quantum computer, the more levels are added to a gate arrangement, with the diagonalization of gate arrangement varying in degree. Higher dimension analysis involves the higher level gate arrangement of the Clifford hierarchy.

### Information quality

Considering the previously mentioned requirement of an intermediate entangled state for quantum teleportation, there needs to be consideration of the purity of this state for information quality. A protection that has been developed involves the use of continuous variable information (rather than a typical discrete variable) creating a superimposed coherent intermediate state. This involves making a phase shift in the received information and then adding a mixing step upon reception using a preferred state, which could be an odd or even coherent state, that will be "conditioned to the classical information of the sender", creating a two mode state that contains the originally sent information.

There have also been developments with teleporting information between systems that already have quantum information in them. Experiments done by Feng, Xu, Zhou et al. have demonstrated that teleportation of a qubit to a photon that already has a qubit's worth of information is possible due to using an optical qubit-ququart entangling gate. This quality can increase computation possibilities as calculations can be done based on previously stored information, allowing for improvements on past calculations.



------------------------------------------------------------------
## Simple Definition:

### Introduction

Quantum teleportation is a way of transferring the quantum state of one quantum system to another using the quantum entanglement between two other systems.

For example, let's assume Alice and Bob share an entangled pair of particles A and B, and Alice wants to teleport the information of another "message" particle M to Bob.

What Alice does is interact her half of the entangled pair, A, she shares with Bob, so that now the message particle M is entangled with Alice's other particle A.

Next Alice measures both of her particles A and M and tells what the outcomes of these measurements are to Bob. Because the message particle M was entangled with Alice's other particle A, and Bob has the information of Alice's measurements, he can use this information to recreate the original message particle M out of the one he has B.

What makes quantum teleportation special is that we didn't need to physically move the message particle M from Alice to Bob.



------------------------------------------------------------------
## Definition Simplified for 10yo:
Imagine you have two friends, Alice and Bob. They have a special connection between them, like a secret code. This code is called entanglement.

Let's say Alice wants to send a secret message to Bob, but she can't just give it to him directly. Instead, she uses her entangled connection with Bob to send the message.

Here's how it works: Alice takes her part of the entangled connection, and she gives it a message. This message is now connected to both parts of the entangled connection - the one Alice has, and the one Bob has.

Then, Alice looks at her part of the connection and sees what the message is. She tells Bob what she sees. Because the message is connected to both parts of the entangled connection, Bob can look at his part and recreate the original message.

The amazing thing about this is that the message didn't move from Alice to Bob. It was like a magic transfer, where the information in the message was sent from Alice's part to Bob's part, using their secret entangled connection.




------------------------------------------------------------------


------------------------------------------------------------------
# Importance sampling



------------------------------------------------------------------
## Definition Simplified for 10yo:
Imagine you have a big box of different colored jelly beans. Each jelly bean represents a different outcome in a game or situation. 

Let's say you want to know how many red jelly beans are in the box, but you can't just count them because the box is very big and you can't see inside. 

One way to solve this problem is to fill a separate, smaller box with the same number of different colored jelly beans as the big box, but this time, make sure there are more red jelly beans in the small box than in the big box. Then, randomly pick a few jelly beans from the small box. 

When you pick a red jelly bean from the small box, you know how many red jelly beans are in the big box, because the small box has more red jelly beans than the big box. This way, you can estimate how many red jelly beans are in the big box without counting them directly.

Importance sampling is similar, but instead of jelly beans, we're talking about numbers or situations. We use a smaller 'box' (called a proposal distribution) that has a similar mix of numbers or situations as the big 'box' (called the target distribution), but with a heavier emphasis on the numbers or situations we're interested in. Then, we pick a few numbers or situations from the smaller 'box' and use them to estimate what's in the big 'box'.




------------------------------------------------------------------


------------------------------------------------------------------
# Absolute truth



------------------------------------------------------------------
## Definition Simplified for 10yo:
You know how we can say things that are true, like "2 + 2 = 4" or "water is wet"? Those are true statements that don't change, no matter what.

Something is called "absolute truth" if it's true all the time and everywhere. It can't be changed by what people think or what happens in a certain situation. For example, if we say "all triangles have three sides," that's an absolute truth because it's true no matter what.

Imagine you're talking to someone, and they say, "I think that's not true." Even if they don't agree, the thing you said is still an absolute truth. It's like saying "red is a color." That's true, no matter what anyone thinks.

But, do you know what's interesting? Some people think that what's true can be different for each person. They think that what's right for one person might not be right for another. That's not what we mean by absolute truth.

There are things that are true just because they are true. We can't prove them with evidence, but they're still true. It's like "a square has four sides." That's true because it's defined that way. 

Some people think that science can't find absolute truths because it's based on evidence, and evidence can be wrong or incomplete. But absolute truths are different. They're based on things that are true by definition, like math or logic.




------------------------------------------------------------------


------------------------------------------------------------------
